In [ ]:
import re
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi stopword remover dan stemmer dari Sastrawi
stop_factory = StopWordRemoverFactory()
stopword_remover = stop_factory.create_stop_word_remover()

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

# Dataset buatan dan berita di JTIK UNM
dokumen_kampus = [
    "Mahasiswa Teknik Komputer UNM Raih Empat Penghargaan pada Pekan Ilmiah Nusantara (PIN 2) 2026.",
    "Dosen Program Studi Teknik Komputer Menjadi Narasumber Guest Lecture di Universitas Negeri Malang.",
    "HIMATIK FT-UNM Torehkan Prestasi Gemilang di PERMI FEST 2026",
    "Pusat komputer kampus sedang memperbarui infrastruktur jaringan kabel fiber optik untuk mempercepat akses internet.",
    "Para peserta praktikum diminta mengumpulkan laporan akhir analisis data sebelum hari Jumat minggu ini."
]

# Fungsi preprocessing teks sesuai urutan di modul
def preprocess_text(text):
    # Case folding
    text_lower = text.lower()

    # Cleaning tanda baca dan angka
    text_clean = re.sub(r'[^a-z\s]', ' ', text_lower)

    # Tokenisasi berbasis spasi
    tokens_raw = text_clean.split()

    # Menghapus stopwords
    text_no_stop = stopword_remover.remove(' '.join(tokens_raw))
    tokens_no_stop = text_no_stop.split()

    # Stemming ke kata dasar
    text_stemmed = stemmer.stem(' '.join(tokens_no_stop))
    tokens_final = text_stemmed.split()

    return tokens_raw, tokens_final

# Menerapkan fungsi preprocessing ke semua dokumen
rekap_data = []

for idx, doc in enumerate(dokumen_kampus, start=1):
    tok_awal, tok_akhir = preprocess_text(doc)

    # Hitung persentase pengurangan kata
    persen_kurang = round((1 - len(tok_akhir) / len(tok_awal)) * 100, 2)

    rekap_data.append({
        'ID': f'Dokumen {idx}',
        'Teks Asli': doc,
        'Token Awal': tok_awal,
        'Token Akhir': tok_akhir,
        'Jml Awal': len(tok_awal),
        'Jml Akhir': len(tok_akhir),
        'Pengurangan (%)': persen_kurang
    })

# Menampilkan perbandingan sebelum & sesudah pada 2 dokumen pertama
print("="*60)
print("PERBANDINGAN HASIL PREPROCESSING (DOKUMEN 1 & 2)")
print("="*60)
for data in rekap_data[:2]:
    print(f"\n[{data['ID']}]")
    print(f"Kalimat Asli : {data['Teks Asli']}")
    print(f"Token Awal   : {data['Token Awal']}")
    print(f"Token Akhir  : {data['Token Akhir']}")
    print(f"Ringkasan    : {data['Jml Awal']} token -> {data['Jml Akhir']} token (Turun {data['Pengurangan (%)']}%)")

# Menampilkan tabel ringkasan token untuk kelima dokumen
df_statistik = pd.DataFrame(rekap_data)[['ID', 'Jml Awal', 'Jml Akhir', 'Pengurangan (%)']]
df_statistik.columns = ['Dokumen', 'Token Awal', 'Token Akhir', 'Pengurangan (%)']

print("\n" + "="*60)
print("TABEL STATISTIK PENGURANGAN TOKEN DOKUMEN")
print("="*60)
print(df_statistik.to_string(index=False))

PERBANDINGAN HASIL PREPROCESSING (DOKUMEN 1 & 2)

[Dokumen 1]
Kalimat Asli : Mahasiswa Teknik Komputer UNM Raih Empat Penghargaan pada Pekan Ilmiah Nusantara (PIN 2) 2026.
Token Awal   : ['mahasiswa', 'teknik', 'komputer', 'unm', 'raih', 'empat', 'penghargaan', 'pada', 'pekan', 'ilmiah', 'nusantara', 'pin']
Token Akhir  : ['mahasiswa', 'teknik', 'komputer', 'unm', 'raih', 'empat', 'harga', 'pekan', 'ilmiah', 'nusantara', 'pin']
Ringkasan    : 12 token -> 11 token (Turun 8.33%)

[Dokumen 2]
Kalimat Asli : Dosen Program Studi Teknik Komputer Menjadi Narasumber Guest Lecture di Universitas Negeri Malang.
Token Awal   : ['dosen', 'program', 'studi', 'teknik', 'komputer', 'menjadi', 'narasumber', 'guest', 'lecture', 'di', 'universitas', 'negeri', 'malang']
Token Akhir  : ['dosen', 'program', 'studi', 'teknik', 'komputer', 'jadi', 'narasumber', 'guest', 'lecture', 'universitas', 'negeri', 'malang']
Ringkasan    : 13 token -> 12 token (Turun 7.69%)

TABEL STATISTIK PENGURANGAN TOKEN DOKUMEN
 

Analisis:

Preprocessing ini terbukti efektif memangkas kata-kata yang tidak perlu (seperti "pada", "di", "untuk") dan memotong simbol serta angka dari teks berita kampus. Selain itu, proses stemming berhasil mengembalikan kata berimbuhan ke bentuk dasarnya (misalnya "menjadi" berubah jadi "jadi"). Efeknya, jumlah token berkurang cukup signifikan sehingga ukuran indeks data jadi jauh lebih ringkas, yang pada akhirnya bikin proses pencarian di sistem Information Retrieval (IR) terasa lebih cepat dan presisi.